# Notebook 1  --  Building Autograd from Scratch

This notebook derives automatic differentiation (backpropagation) from first
principles, starting from the scalar chain rule and extending it to matrix
calculus.  Every formula is verified numerically against finite differences.

---

## 1. The Scalar Chain Rule

Given a composition $f(g(x))$, the chain rule states:

$$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial g} \cdot \frac{\partial g}{\partial x}$$

In a deep network we have a long chain of compositions:

$$L = \ell(\sigma(W_2 \sigma(W_1 x)))$$

Backpropagation is simply repeated application of the chain rule, accumulating
gradients from the output backwards towards the inputs.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from gnn.autograd.tensor import Tensor

## 2. A Single Scalar Example

Let's compute $L = (wx + b)^2$ and verify $\partial L / \partial w$.

In [ ]:
# Manual scalar computation
w_val, x_val, b_val = 2.0, 3.0, -1.0

# Forward pass
z = w_val * x_val + b_val   # z = 5.0
L = z ** 2                  # L = 25.0

# Analytical gradient: dL/dw = 2z * x
dL_dw_analytic = 2 * z * x_val

# Numerical gradient (finite difference)
eps = 1e-5
z_plus  = (w_val + eps) * x_val + b_val
z_minus = (w_val - eps) * x_val + b_val
dL_dw_numeric = (z_plus**2 - z_minus**2) / (2 * eps)

print(f"Analytic  dL/dw = {dL_dw_analytic:.6f}")
print(f"Numerical dL/dw = {dL_dw_numeric:.6f}")
print(f"Match: {np.isclose(dL_dw_analytic, dL_dw_numeric)}")

## 3. Computation Graphs

Modern autograd frameworks represent computations as **directed acyclic graphs (DAGs)**.
Each node is an operation; edges carry tensors.  Backprop traverses this graph
in reverse topological order.

The cell below draws the graph for a two-layer MLP: `L = NLL(log_softmax(relu(X @ W1) @ W2))`.

Our `Tensor` class records this graph via `_prev` and `_backward`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Node layout: (label, x, y, colour)
nodes = [
    ("X",            0.0, 2.0, "#4C72B0"),
    ("W1",           0.0, 0.5, "#4C72B0"),
    ("@",            1.5, 1.25, "#DD8452"),
    ("ReLU",         2.8, 1.25, "#DD8452"),
    ("W2",           2.8, 0.0, "#4C72B0"),
    ("@",            4.1, 0.75, "#DD8452"),
    ("log\nsoftmax", 5.3, 0.75, "#DD8452"),
    ("NLL\nloss",    6.5, 0.75, "#DD8452"),
    ("y\n(labels)",  6.5, 2.0, "#55A868"),
]

edges = [
    (0, 2), (1, 2),
    (2, 3),
    (3, 5), (4, 5),
    (5, 6),
    (6, 7), (8, 7),
]

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.set_xlim(-0.7, 7.5)
ax.set_ylim(-0.7, 2.7)
ax.axis("off")

R = 0.38

def _unit(sx, sy, dx, dy):
    d = ((dx-sx)**2 + (dy-sy)**2)**0.5
    return (dx-sx)/d, (dy-sy)/d

for src, dst in edges:
    sx, sy = nodes[src][1], nodes[src][2]
    dx, dy = nodes[dst][1], nodes[dst][2]
    ux, uy = _unit(sx, sy, dx, dy)
    ax.annotate("",
        xy=(dx - R*ux, dy - R*uy),
        xytext=(sx + R*ux, sy + R*uy),
        arrowprops=dict(arrowstyle="-|>", color="#888", lw=1.5, mutation_scale=16))

for label, x, y, colour in nodes:
    ax.add_patch(plt.Circle((x, y), R, facecolor=colour, edgecolor="white",
                             linewidth=1.5, zorder=3))
    ax.text(x, y, label, ha="center", va="center", fontsize=8.5,
            fontweight="bold", color="white", zorder=4, linespacing=1.3)

ax.legend(handles=[
    mpatches.Patch(color="#4C72B0", label="Leaf (input / weight)"),
    mpatches.Patch(color="#DD8452", label="Operation node"),
    mpatches.Patch(color="#55A868", label="Leaf (target, no grad)"),
], loc="upper right", fontsize=9, framealpha=0.85)

ax.annotate("", xy=(6.9, -0.30), xytext=(-0.1, -0.30),
    arrowprops=dict(arrowstyle="-|>", color="#2196F3", lw=2.0))
ax.text(3.4, -0.47, "forward pass", ha="center", color="#2196F3", fontsize=9)

ax.annotate("", xy=(-0.1, -0.50), xytext=(6.9, -0.50),
    arrowprops=dict(arrowstyle="-|>", color="#E53935", lw=2.0))
ax.text(3.4, -0.67, "backward pass  (reverse topological order)",
        ha="center", color="#E53935", fontsize=9)

plt.title("Computation graph for  L = NLL(log_softmax(ReLU(X @ W1) @ W2))",
          fontsize=10, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate Tensor graph building
w = Tensor(np.array(2.0), requires_grad=True)
x = Tensor(np.array(3.0))
b = Tensor(np.array(-1.0), requires_grad=True)

# Note: scalar mul isn't in our Tensor (it targets matrices);
# we demo the matrix ops instead  --  see below.

## 4. Matrix Calculus: The Key Rules

For a linear layer $Z = XW$ where $X \in \mathbb{R}^{N \times d}$ and
$W \in \mathbb{R}^{d \times k}$, and scalar loss $L$:

$$\frac{\partial L}{\partial W} = X^\top \frac{\partial L}{\partial Z}
\qquad
\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Z} W^\top$$

**Derivation**  --  using the trace trick and $dL = \text{tr}\left(\left(\frac{\partial L}{\partial Z}\right)^\top dZ\right)$:

$$dL = \text{tr}\left(G^\top d(XW)\right) = \text{tr}\left(G^\top X\, dW\right)
     = \text{tr}\left((X^\top G)^\top dW\right)$$

So $\partial L / \partial W = X^\top G$ where $G = \partial L / \partial Z$.

In [ ]:
np.random.seed(0)
N, d, k = 5, 4, 3

X_data = np.random.randn(N, d)
W_data = np.random.randn(d, k)

X = Tensor(X_data.copy())
W = Tensor(W_data.copy(), requires_grad=True)

# Forward: Z = X @ W  ->  L = sum(Z)
Z = X @ W
# Scalar loss: L = mean(Z)
# For simplicity inject upstream gradient G = ones/N manually
Z.grad = np.ones((N, k)) / (N * k)
Z._backward()

analytical_dW = W.grad.copy()

# Numerical gradient
eps = 1e-5
numerical_dW = np.zeros_like(W_data)
for i in range(d):
    for j in range(k):
        Wp = W_data.copy(); Wp[i,j] += eps
        Wm = W_data.copy(); Wm[i,j] -= eps
        Lp = (X_data @ Wp).mean()
        Lm = (X_data @ Wm).mean()
        numerical_dW[i,j] = (Lp - Lm) / (2 * eps)

print("Max absolute error (dL/dW):", np.abs(analytical_dW - numerical_dW).max())
print("Gradients match:", np.allclose(analytical_dW, numerical_dW, atol=1e-6))

## 5. ReLU Gradient

$$\text{ReLU}(x) = \max(0, x)$$

$$\frac{\partial \text{ReLU}(x)}{\partial x} = \mathbf{1}[x > 0]$$

Element-wise operation -> element-wise gradient (Hadamard product with the
upstream gradient).

In [ ]:
np.random.seed(1)
A_data = np.random.randn(4, 3)

A = Tensor(A_data.copy(), requires_grad=True)
R = A.relu()
R.grad = np.ones_like(A_data)
R._backward()
analytical = A.grad.copy()

# Numerical
eps = 1e-5
numerical = np.zeros_like(A_data)
for i in range(A_data.shape[0]):
    for j in range(A_data.shape[1]):
        Ap = A_data.copy(); Ap[i,j] += eps
        Am = A_data.copy(); Am[i,j] -= eps
        numerical[i,j] = (np.maximum(0,Ap).sum() - np.maximum(0,Am).sum()) / (2*eps)

print("Max absolute error (ReLU grad):", np.abs(analytical - numerical).max())
print("Gradients match:", np.allclose(analytical, numerical, atol=1e-6))

## 6. Softmax + Cross-Entropy Gradient

The log-softmax of logit vector $z$ is:

$$\text{log-softmax}(z_i) = z_i - \log \sum_j e^{z_j}$$

Its Jacobian is: $J_{ij} = \delta_{ij} - \text{softmax}(z_j)$

Combined with NLL loss $L = -\text{log-softmax}(z_y)$ (for true class $y$),
the gradient simplifies beautifully:

$$\frac{\partial L}{\partial z_i} = \text{softmax}(z_i) - \mathbf{1}[i = y]$$

"Predicted probability minus true label"  --  the network learns by reducing the
gap between what it predicts and the ground truth.

In [ ]:
np.random.seed(2)
logits_data = np.random.randn(6, 7)  # 6 nodes, 7 classes
targets = np.array([3, 1, 6, 0, 4, 2])
mask = np.ones(6, dtype=bool)

logits = Tensor(logits_data.copy(), requires_grad=True)
lsm    = logits.log_softmax()
loss   = lsm.nll_loss(targets, mask)
loss.backward()

analytical = logits.grad.copy()

# Numerical
eps = 1e-5
numerical = np.zeros_like(logits_data)
for i in range(logits_data.shape[0]):
    for j in range(logits_data.shape[1]):
        def forward(ld):
            shifted = ld - ld.max(axis=1, keepdims=True)
            lsm_v = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
            return -lsm_v[mask, targets[mask]].mean()
        Lp = logits_data.copy(); Lp[i,j] += eps
        Lm = logits_data.copy(); Lm[i,j] -= eps
        numerical[i,j] = (forward(Lp) - forward(Lm)) / (2*eps)

print("Max absolute error (log-softmax + NLL grad):", np.abs(analytical - numerical).max())
print("Gradients match:", np.allclose(analytical, numerical, atol=1e-5))

## 7. Full End-to-End Gradient Check

A two-layer MLP (no graph structure)  --  verifies the complete backward pass.

In [ ]:
from gnn.autograd.tensor import Tensor, relu, log_softmax, nll_loss

np.random.seed(42)
N, d_in, d_h, d_out = 8, 10, 6, 4

X_data = np.random.randn(N, d_in)
W1_data = np.random.randn(d_in, d_h) * 0.1
W2_data = np.random.randn(d_h, d_out) * 0.1
y_data  = np.random.randint(0, d_out, N)
mask    = np.ones(N, dtype=bool)

def forward_np(X, W1, W2, y, mask):
    """Pure numpy forward for finite differences."""
    H = np.maximum(0, X @ W1)
    Z = H @ W2
    shifted = Z - Z.max(axis=1, keepdims=True)
    lsm = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    return -lsm[mask, y[mask]].mean()

# Autograd gradient for W1
X  = Tensor(X_data.copy())
W1 = Tensor(W1_data.copy(), requires_grad=True)
W2 = Tensor(W2_data.copy(), requires_grad=True)

H  = (X @ W1).relu()
Z  = H @ W2
lsm_t = Z.log_softmax()
loss_t = lsm_t.nll_loss(y_data, mask)
loss_t.backward()

analytical_W1 = W1.grad.copy()

# Numerical gradient for W1
eps = 1e-5
numerical_W1 = np.zeros_like(W1_data)
for i in range(W1_data.shape[0]):
    for j in range(W1_data.shape[1]):
        Wp = W1_data.copy(); Wp[i,j] += eps
        Wm = W1_data.copy(); Wm[i,j] -= eps
        numerical_W1[i,j] = (forward_np(X_data,Wp,W2_data,y_data,mask) -
                              forward_np(X_data,Wm,W2_data,y_data,mask)) / (2*eps)

max_err = np.abs(analytical_W1 - numerical_W1).max()
print(f"Max gradient error (W1): {max_err:.2e}")
print(f"Gradient check passed: {max_err < 1e-5}")

## 8. Topological Sort Visualised

The backward pass traverses nodes in **reverse topological order**  --  every
node is visited only after all its consumers have been visited.

Below we print the order in which `backward()` calls `_backward()` for our
two-layer network.

In [ ]:
# Instrument Tensor to trace backward order
order = []

X  = Tensor(X_data[:3])
W1 = Tensor(W1_data.copy(), requires_grad=True)
W2 = Tensor(W2_data.copy(), requires_grad=True)

H  = (X @ W1).relu()
Z  = H @ W2
lsm_t = Z.log_softmax()
loss_t = lsm_t.nll_loss(y_data[:3], np.ones(3, dtype=bool))

# Reconstruct topo order (mirrors Tensor.backward internals)
topo, visited = [], set()
def build_topo(node):
    if id(node) not in visited:
        visited.add(id(node))
        for child in node._prev:
            build_topo(child)
        topo.append(node)
build_topo(loss_t)

print("Forward graph (leaf -> output):")
for i, node in enumerate(topo):
    print(f"  {i:2d}  op='{node._op or 'leaf':12s}'  shape={str(node.shape):15s}  req_grad={node.requires_grad}")

print("\nBackward traversal order (reversed):")
for i, node in enumerate(reversed(topo)):
    print(f"  {i:2d}  op='{node._op or 'leaf':12s}'")

## Summary

| Operation | Forward | Backward (given upstream grad $G$) |
|-----------|---------|------------------------------------|
| $C = A @ B$ | $A \cdot B$ | $\partial L/\partial A = G B^\top$, $\partial L/\partial B = A^\top G$ |
| $C = A + B$ | $A + B$ | $\partial L/\partial A = G$, $\partial L/\partial B = G$ (sum over broadcast) |
| $C = \text{ReLU}(A)$ | $\max(0, A)$ | $G \odot \mathbf{1}[A > 0]$ |
| $C = \text{log-softmax}(A)$ | $A - \log\sum e^A$ | $G - \text{softmax}(A) \cdot \sum G$ |
| $L = \text{NLL}(Z, y)$ | $-Z_{n,y_n}$ mean | $-1/N$ at true class positions |

These five rules are all that is needed to train a two-layer GCN.